# Seasonality and Events on One M5 Item, Store by Store

This notebook fits one item, `FOODS_3_586`, in each of the 10 M5 stores with the #118 default M5 model: a level state
and six weekday states that drift, a Yeo-Johnson target and half-normal priors on its sigmas. It then adds an annual
Fourier pair and one column per M5 event, as covariates with fixed coefficients, and compares four fits:

| Fit | Added columns | Prior of each added coefficient |
|---|---|---|
| **no covariates** | none | none |
| **loose** | annual Fourier and events | $N(0, 1)$ |
| **tight** | the same columns | $N(0, 0.1^2)$ |
| **events only** | events, without the annual Fourier pair | $N(0, 1)$ |

Every store is fitted on its own, with its own coefficients. The last section splits the forecast of one store into its
level, day of week, annual Fourier and events. This is the univariate baseline of a model that shares the coefficients
across stores. It is not a scored stage. It needs the raw M5 files (`m5_fetch.py`) and the `m5` extra:
`pip install -e ".[m5]"`. It is committed without outputs.

## 1. Libraries and data

The notebook reads the 10 store series of one item from `sales_train_evaluation.csv`, over the same 728 days as
`ssp_m5_aggregate_forecast_seasonality`: 700 days to fit on, d_1186 to d_1885, and 28 to score, d_1886 to d_1913, the
pre-validation window of #125. It reads no column after d_1913, so M5's validation window, d_1914 to d_1941, which
#143 reads only once per candidate, never enters memory.

In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bunobee.models.m5.m5_ssp_optim import SigmaPrior, fit_one_series_opt, predict_one_series_opt
from bunobee.models.m5.transforms import PowerTransform
from bunobee.regression import make_fourier_series, make_periodic_dummies
from m5_common import DATA_DIR  # Found because Jupyter runs from this folder, playground/m5_prototype.

plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 10,
        "axes.titlesize": 11,
        "lines.linewidth": 1.5,
    }
)

# One color per fit, the same in every figure. Actual sales are black.
FIT_COLORS = {"no covariates": "#2a78d6", "loose": "#eb6834", "tight": "#1baf7a", "events only": "#eda100"}
ACTUAL_COLOR = "#0b0b0b"
CONTEXT_COLOR = "#b5b4ad"

In [ ]:
ITEM = "FOODS_3_586"
N_FIT = 700
HORIZON = 28
FIRST_DAY = 1186  # d_1186, the first fit day
LAST_DAY = FIRST_DAY + N_FIT + HORIZON - 1  # d_1913, the last day this notebook reads

day_columns = [f"d_{d}" for d in range(FIRST_DAY, LAST_DAY + 1)]
raw = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv", usecols=["item_id", "store_id", *day_columns])
raw = raw[raw["item_id"] == ITEM].set_index("store_id").drop(columns="item_id").sort_index()
STORES = raw.index.tolist()
sales = raw.to_numpy(dtype=float).T  # (days, stores)

days = pd.read_csv(DATA_DIR / "calendar.csv", keep_default_na=False).set_index("d").loc[day_columns]
dates = pd.DatetimeIndex(days["date"])  # One row of `days` per date, with the calendar columns.

assert sales.shape == (N_FIT + HORIZON, 10) and LAST_DAY == 1913
print(f"fit:     {dates[0].date()} to {dates[N_FIT - 1].date()}, d_{FIRST_DAY} to d_{FIRST_DAY + N_FIT - 1}")
print(f"holdout: {dates[N_FIT].date()} to {dates[-1].date()}, d_{FIRST_DAY + N_FIT} to d_{LAST_DAY}")
print(f"stores:  {STORES}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 5), sharex=True)
for j, (ax, store) in enumerate(zip(axes.flat, STORES)):
    ax.plot(dates, sales[:, j], color=ACTUAL_COLOR, lw=0.6)
    ax.axvspan(dates[N_FIT], dates[-1], color="#f0efe9", zorder=0)
    ax.set_title(store)
axes.flat[0].set_ylabel("units")
axes.flat[5].set_ylabel("units")
fig.suptitle(f"{ITEM}: daily unit sales, with the holdout shaded")
fig.tight_layout()
plt.show()

## 2. The design

The base columns are the default's: a column of ones for the level, then Tuesday to Sunday with Monday dropped. Those
are in every fit. The covariates are the two groups below, built from the calendar. They are the same for every store.
The `events only` fit uses the second group alone:

- **Annual:** the sine and cosine of period 365.25 days, the first annual Fourier pair, with step 0 at the first day of
  the M5 calendar, 2011-01-29.
- **Events:** one 0/1 column per name in `event_name_1` and `event_name_2`.

In [ ]:
base = make_periodic_dummies(len(dates), period=7, phase=int(dates[0].dayofweek), intercept=True)
n_base = base.shape[1]

annual = pd.DataFrame(
    make_fourier_series(LAST_DAY, period=365.25, order=1)[FIRST_DAY - 1 :], index=days.index, columns=["sin", "cos"]
)
event_names = np.setdiff1d(days[["event_name_1", "event_name_2"]], [""]).tolist()
events = pd.DataFrame({n: (days["event_name_1"] == n) | (days["event_name_2"] == n) for n in event_names}).astype(float)
calendar = pd.concat({"annual": annual, "event": events}, axis=1)

COV_GROUPS = np.array(calendar.columns.get_level_values(0))
COV_NAMES = np.array(calendar.columns.get_level_values(1))
covariates = calendar.to_numpy()  # (days, n_covariates)
N_COVARIATES = covariates.shape[1]

print(f"base columns: {n_base} (level, then Tuesday to Sunday)")
print(f"covariates:   {N_COVARIATES}")
pd.Series(COV_GROUPS).value_counts(sort=False).rename("columns").to_frame()

The events, each with its number of fit days and holdout days. The holdout has **no event day**, so the event columns
reach its forecasts only through what they take away from the other states, and it can't say whether their
coefficients forecast well. Some events are seen once in the fit days. Unlike the aggregate panel, the item series are
not repaired for the Christmas closure: the stores are closed on 25 December, so the Christmas column has a real drop
to learn, but only two days to learn it from in each store.

In [ ]:
event_days = pd.DataFrame(
    {"fit days": events.iloc[:N_FIT].sum().astype(int), "holdout days": events.iloc[N_FIT:].sum().astype(int)}
)
print(f"event days in the holdout: {event_days['holdout days'].sum()}")
christmas = events["Christmas"].to_numpy()[:N_FIT] == 1
print(f"units sold on the {christmas.sum()} Christmas fit days, over all stores: {sales[:N_FIT][christmas].sum():.0f}")
event_days.T

## 3. Fits with no covariates, a loose prior and a tight prior

The model, on the standardized Yeo-Johnson scale of a store's series, is

$$y_t = \ell_t + w_t^\top d_t + \beta^\top x_t + \varepsilon_t, \qquad \beta_j \sim N(0, \tau^2),$$

with the level $\ell_t$, the weekday states $w_t$ on the weekday columns $d_t$, and the fixed coefficients $\beta$ on
the covariates $x_t$. $\tau$ is the only thing that changes between the loose and the tight fit:

- **Loose, $\tau = 1$:** the target has a standard deviation of 1 over the fit days, so a single coefficient may be as
  large as the whole series' variation.
- **Tight, $\tau = 0.1$:** one tenth of that. About 95% of the prior's mass lies within $\pm 0.2$ of the target's
  standard deviation, well inside the daily noise. A coefficient needs many days behind it to move far from zero.
- **Events only, $\tau = 1$:** the loose prior on the event columns alone. Next to the loose fit it shows what the
  annual Fourier pair adds, since the two differ only in those two columns.

`fit_one_series_opt` has no argument for $\tau$, since it gives every state `a0 = 0` and `P0 = 1`, but it needs none:
a column $\tau x$ with a coefficient $b \sim N(0, 1)$ is the same model as a column $x$ with a coefficient
$\beta = \tau b \sim N(0, \tau^2)$. So the notebook scales the covariate columns by $\tau$ and multiplies the fitted
coefficients back by $\tau$.

Each store gets its own MAP fit on its 700 fit days: Adam chooses the sigmas against the filter's likelihood and the
filter updates the coefficients day by day. The forecast uses the states filtered up to the last fit day.

In [ ]:
TRANSFORM = PowerTransform("yeo-johnson")
# The half-normal priors of the #118 default (HALF_NORMAL_PRIORS in playground/m5_prototype/m5_models.py).
SIGMA_PRIORS = {"sigma_h_prior": SigmaPrior("half_normal", (1.0,)), "sigma_q_prior": SigmaPrior("half_normal", (0.05,))}

# The prior scale tau of each fit's covariates, and the groups of covariate columns it uses. tau=None has none.
FITS = {
    "no covariates": {"tau": None, "groups": ()},
    "loose": {"tau": 1.0, "groups": ("annual", "event")},
    "tight": {"tau": 0.1, "groups": ("annual", "event")},
    "events only": {"tau": 1.0, "groups": ("event",)},
}


def kept(name):
    # Which of the covariate columns a fit uses.
    return np.isin(COV_GROUPS, FITS[name]["groups"])


def design(name, scaled=True):
    # The design of a fit over every day, with its covariates scaled by tau. scaled=False leaves them unscaled.
    tau = FITS[name]["tau"]
    if tau is None:
        return base
    return np.column_stack([base, (tau if scaled else 1.0) * covariates[:, kept(name)]])


def fit_store(j, name):
    # One MAP fit of store j. `states` are the filtered states of every fit day, with the coefficients on the unscaled
    # columns (beta = tau * b).
    tau = FITS[name]["tau"]
    Z = design(name).astype(np.float32)
    fit = fit_one_series_opt(
        sales[:N_FIT, j],
        Z=jnp.asarray(Z[:N_FIT]),
        transform=TRANSFORM,
        fixed_states=None if tau is None else np.arange(Z.shape[1]) >= n_base,
        **SIGMA_PRIORS,
    )
    states = fit["at"].copy()
    states[:, n_base:] *= tau or 1.0
    return {
        "forecast": predict_one_series_opt(fit, Z_future=Z[N_FIT:]),
        "states": states,
        "sigma_h": fit["sigma_h"],
        "sigma_q_level": fit["sigma_q"][0],
        "loss": fit["final_loss"],
    }


def fit_stores(name):
    # Every store under one fit, each output stacked with the store first.
    by_store = [fit_store(j, name) for j in range(len(STORES))]
    return {key: np.stack([fit[key] for fit in by_store]) for key in by_store[0]}


results = {name: fit_stores(name) for name in FITS}

MAPE and bias on the 28 holdout days, by store. Bias is the error of the 28-day sum. The seasonal-naive column repeats
the last fit week, the baseline of `tests/test_m5_accuracy.py`.

In [ ]:
actual = sales[N_FIT:]  # (horizon, stores)
naive = sales[N_FIT - 7 : N_FIT][np.arange(HORIZON) % 7]
assert (actual > 0).all(), "MAPE needs a nonzero actual on every holdout day."


def mape(forecast):
    return 100 * np.mean(np.abs(forecast - actual) / actual, axis=0)


def bias(forecast):
    return 100 * (forecast.sum(axis=0) / actual.sum(axis=0) - 1)


forecasts = {name: result["forecast"].T for name, result in results.items()}  # (horizon, stores)
mape_table = pd.DataFrame({name: mape(f) for name, f in forecasts.items()} | {"seasonal naive": mape(naive)}, STORES)
bias_table = pd.DataFrame({name: bias(f) for name, f in forecasts.items()}, STORES)

accuracy = pd.concat({"MAPE %": mape_table, "bias %": bias_table}, axis=1)
accuracy.loc["mean"] = accuracy.mean()
accuracy.round(2)

The sigmas and the loss of each fit, as the median over the stores, and the loss summed over them. `loss` is the fit's
negative log posterior of the sigmas, which sees only the fit days.

In [ ]:
pd.DataFrame(
    {
        name: {
            "sigma_h": np.median(result["sigma_h"]),
            "level drift": np.median(result["sigma_q_level"]),
            "summed loss": result["loss"].sum(),
        }
        for name, result in results.items()
    }
).T.round(3)

In [ ]:
context = 14  # Fit days drawn before the holdout.
fig, axes = plt.subplots(2, 5, figsize=(15, 5.5), sharex=True)
for j, (ax, store) in enumerate(zip(axes.flat, STORES)):
    ax.plot(dates[N_FIT - context :], sales[N_FIT - context :, j], color=ACTUAL_COLOR, lw=1.2, label="actual")
    for name, forecast in forecasts.items():
        ax.plot(dates[N_FIT:], forecast[:, j], color=FIT_COLORS[name], label=name)
    ax.axvline(dates[N_FIT], color=CONTEXT_COLOR, lw=0.8)
    ax.set_title(store)
    ax.xaxis.set_major_locator(mdates.DayLocator(bymonthday=(1, 15)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d"))
axes.flat[0].set_ylabel("units")
axes.flat[5].set_ylabel("units")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=5, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 4. Decomposition into four components

Each fit's forecast mean, on the standardized target scale, is the sum of four parts:

- **level:** the filtered level state $\ell_t$.
- **day of week:** the six weekday states on their columns, $w_t^\top d_t$.
- **annual Fourier:** the sine and cosine columns times their two coefficients.
- **events:** the event columns times their coefficients, summed over all events into one line.

Over the fit days the level and weekday states are the filtered path. The covariate coefficients are fixed, so the
annual and event parts use their values on the last fit day throughout. Over the holdout the states are held at their
last fit day values, which is what the forecast does, so the four parts add up to the forecast mean there. The
`no covariates` fit has only the first two parts, and `events only` has no annual part. The cell decomposes every store
and draws the one in `STORE`.

In [ ]:
STORE = "CA_1"


def components(name, j):
    # The components of one fit of store j over every day, on the target scale. The covariate parts are absent
    # without covariates.
    states = results[name]["states"][j]
    full = np.vstack([states, np.tile(states[-1], (HORIZON, 1))])  # Hold the last fit day's states over the holdout.
    out = {"level": full[:, 0], "day of week": np.einsum("ti,ti->t", base[:, 1:], full[:, 1:n_base])}
    beta = states[-1, n_base:]
    groups = COV_GROUPS[kept(name)]
    for group, label in (("annual", "annual Fourier"), ("event", "events")):
        if group in groups:
            out[label] = covariates[:, kept(name)][:, groups == group] @ beta[groups == group]
    return pd.DataFrame(out, index=dates)


parts = {name: [components(name, j) for j in range(len(STORES))] for name in FITS}

# The parts add up to the forecast mean of the design, Z @ states, on the holdout days.
for name in FITS:
    Z = design(name, scaled=False)[N_FIT:]  # Unscaled columns, with the unscaled coefficients.
    for j, store in enumerate(STORES):
        total = parts[name][j].iloc[N_FIT:].sum(axis=1)
        assert np.allclose(total, Z @ results[name]["states"][j, -1], atol=1e-4), store

In [ ]:
labels = ["level", "day of week", "annual Fourier", "events"]
j = STORES.index(STORE)
fig, axes = plt.subplots(len(labels), 1, figsize=(12, 9), sharex=True)
for ax, label in zip(axes, labels):
    for name in FITS:
        if label in parts[name][j]:
            ax.plot(dates, parts[name][j][label], color=FIT_COLORS[name], lw=0.9, label=name)
    ax.axvspan(dates[N_FIT], dates[-1], color="#f0efe9", zorder=0)
    ax.set_title(label)
    ax.set_ylabel("target scale")
axes[0].legend(frameon=False, ncol=4)
axes[0].set_title(f"{STORE}: level. With covariates it is the level of a Monday with no event, not of an average day.")
fig.tight_layout()
plt.show()

How much each component of `STORE` moves, as the standard deviation of its values over the fit days. A tight prior
shrinks the annual and event parts toward zero and leaves the level and the weekday states to carry the variation.

In [ ]:
spread = pd.DataFrame({name: parts[name][j].iloc[:N_FIT].std() for name in FITS}).reindex(labels)
spread.round(3)